In [9]:
import json
from openai import OpenAI
from dotenv import load_dotenv

In [15]:
def get_order_status(order_id: str) -> dict:
    orders = {
        "ORD-101": {"item": "Oat Latte", "status": "Ready to pickup"},
        "ORD-102": {"item": "Cappuccino", "status": "In progress"},
    }
    order = orders.get(order_id.upper())
    
    if not order:
        return {"found": False, "order_id": order_id}
    return {
        "found": True,
        "order_id": order_id,
        "item": order["item"],
        "status": order["status"]
    }

In [4]:
get_order_status("ORD-106")

{'found': False, 'order_id': 'ORD-106'}

In [5]:
get_order_status("ORD-101")

{'found': True,
 'order_id': 'ORD-101',
 'item': 'Oat Latte',
 'status': 'Ready to pickup'}

### Create a schema for tools

In [19]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_order_status",
            "description": "Get the status of an order by its ID",
            "parameters": {
                "type": "object",
                "properties": {
                    "order_id": {
                        "type": "string",
                        "description": "The ID of the order to check the status for."
                    }
                },
                "required": ["order_id"]
            }
        }
    }
]

In [20]:
load_dotenv()
client = OpenAI()

In [27]:
messages: list[dict[str, str]] = [
    {
        "role": "user",
        "content": "Check the status of order ORD-102"
    }
]

In [28]:
response = client.chat.completions.create(
    model="gpt-5.4-nano",
    messages=messages,
    tools=tools
)

assistant_message = response.choices[0].message
messages.append(assistant_message)


In [29]:
if assistant_message.tool_calls:
    for tool_call in assistant_message.tool_calls:
        if tool_call.function.name == "get_order_status":
            args = json.loads(tool_call.function.arguments)
            result = get_order_status(args["order_id"])
            
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(result)
            })
    final_response = client.chat.completions.create(
        model="gpt-5.4-nano",
        messages=messages,
        tools=tools
    )
    print("Final response:", final_response.choices[0].message.content)
    
else:
    print("Assistant response:", assistant_message.content)


Final response: Order **ORD-102** status: **In progress** (Item: **Cappuccino**).
